# Sea Ice Contour Plots for Multiple CESM Runs

In [ ]:
import glob
import os

import xarray as xr
import numpy as np
import yaml

import cupid_utils as cu

xr.set_options(file_cache_maxsize=1024);

In [ ]:
# These parameters are needed when running interactively

CESM_output_dir = ""  # "/glade/campaign/cesm/development/cross-wg/diagnostic_framework/CESM_output_for_testing"
ts_dir = None  # "/glade/campaign/cesm/development/cross-wg/diagnostic_framework/CESM_output_for_testing"

case_names = []
case_nicknames = []
start_dates = []
end_dates = []

obs_data_dir = ""  # "/glade/campaign/cesm/development/cross-wg/diagnostic_framework/CUPiD_obs_data"
grid_file = ""  # "/glade/campaign/cesm/community/omwg/grids/tx2_3v2_grid.nc"
climo_nyears = 35

serial = False  # use dask LocalCluster

lc_kwargs = {}

In [ ]:
# Set some parameter defaults
if ts_dir is None:
    ts_dir = CESM_output_dir

In [ ]:
# When running interactively, cupid_run should be set to 0 for
# a DASK cluster

cupid_run = 1

if cupid_run == 1:

    from dask.distributed import Client, LocalCluster

    # Spin up cluster (if running in parallel)
    client = None
    if not serial:
        cluster = LocalCluster(**lc_kwargs)
        client = Client(cluster)

else:

    from dask.distributed import Client
    from dask_jobqueue import PBSCluster

    cluster = PBSCluster(
        cores=16,
        processes=16,
        memory="100GB",
        account="P93300065",
        queue="casper",
        walltime="02:00:00",
    )

    client = Client(cluster)

    cluster.scale(1)

    print(cluster)
print(client)

In [ ]:
# set up dictionary for all cases listed

cice_hist_vars = ["aice", "hi", "hs"]

case_info = {}
for i, (case_name, case_nickname, start_date, end_date, tsdir) in enumerate(
    zip(case_names, case_nicknames, start_dates, end_dates, ts_dir)
):
    # Only open the variables used here; opening every tseries file can exceed
    # xarray's file cache and cause "NetCDF: Not a valid ID" errors.
    ts_files = sorted(
        f
        for var in cice_hist_vars
        for f in glob.glob(
            os.path.join(
                tsdir,
                case_name,
                "ice",
                "proc",
                "tseries",
                f"{case_name}.cice.h.{var}.*.nc",
            )
        )
    )
    ds = xr.open_mfdataset(
        ts_files,
        data_vars="minimal",
        compat="override",
        coords="minimal",
    ).sel(time=slice(start_date, end_date))

    ds_ann = ds.resample(time="YS").mean(dim="time")
    climo_n_years = min(
        climo_nyears, len(ds_ann.time)
    )  ## had to name this something different than global climo_nyears set in params

    case_info[case_name] = {
        "case_nickname": case_nickname,
        "start_date": start_date,
        "end_date": end_date,
        "ds": ds,
        "ds_ann": ds_ann,
        "climo_nyears": climo_n_years,
    }

In [ ]:
# Read in cases. The ADF timeseries are needed here.

ds_grid = xr.open_dataset(grid_file)
if "nx" in ds_grid.dims:
    ds_grid = ds_grid.rename({"nx": "ni", "ny": "nj"})

if "TLAT" in ds_grid:
    TLAT = ds_grid["TLAT"]
    TLON = ds_grid["TLONG"]
    tarea = ds_grid["TAREA"] * 1.0e-4
    angle = ds_grid["ANGLE"]
else:
    TLAT = ds_grid["tlat"]
    TLON = ds_grid["tlon"]
    tarea = ds_grid["tarea"]
    angle = ds_grid["angle"]

yml_dir = os.path.join(cu.__path__[0], "..", "input_files", "ice")
with open(os.path.join(yml_dir, "cice_masks.yml"), "r") as file:
    cice_masks = yaml.safe_load(file)

with open(os.path.join(yml_dir, "cice_vars.yml"), "r") as file:
    cice_vars = yaml.safe_load(file)

path_HadleyOI = os.path.join(obs_data_dir, "ice", "analysis_datasets", "1x1d/")

In [ ]:
from cupid_utils.ice import plot_diff
from cupid_utils.ice import vect_diff

# Northern Hemisphere Contour Plots

In [ ]:
var = "aice"
levels = np.array(cice_vars[var][0]["levels"])
title = "March " + cice_vars[var][1]["title"]
month = 3

plot_diff(
    case_names,
    case_info,
    var,
    levels,
    title,
    "N",
    month,
    TLAT,
    TLON,
    path_HadleyOI,
)

levels = np.array(cice_vars[var][0]["levels"])
title = "September " + cice_vars[var][1]["title"]
month = 9

plot_diff(
    case_names,
    case_info,
    var,
    levels,
    title,
    "N",
    month,
    TLAT,
    TLON,
    path_HadleyOI,
)

In [ ]:
var = "hi"
levels = np.array(cice_vars[var][0]["levels"])
title = "March " + cice_vars[var][1]["title"]
month = 3

plot_diff(
    case_names,
    case_info,
    var,
    levels,
    title,
    "N",
    month,
    TLAT,
    TLON,
    path_HadleyOI,
)

levels = np.array(cice_vars[var][0]["levels"])
title = "September " + cice_vars[var][1]["title"]
month = 9

plot_diff(
    case_names,
    case_info,
    var,
    levels,
    title,
    "N",
    month,
    TLAT,
    TLON,
    path_HadleyOI,
)

In [ ]:
var = "hs"
levels = np.array(cice_vars[var][0]["levels"])
title = "March " + cice_vars[var][1]["title"]
month = 3

plot_diff(
    case_names,
    case_info,
    var,
    levels,
    title,
    "N",
    month,
    TLAT,
    TLON,
    path_HadleyOI,
)

levels = np.array(cice_vars[var][0]["levels"])
title = "August " + cice_vars[var][1]["title"]
month = 8

plot_diff(
    case_names,
    case_info,
    var,
    levels,
    title,
    "N",
    month,
    TLAT,
    TLON,
    path_HadleyOI,
)

# Southern Hemisphere Contour Plots

In [ ]:
var = "aice"
levels = np.array(cice_vars[var][0]["levels"])
title = "February " + cice_vars[var][1]["title"]
month = 2

plot_diff(
    case_names,
    case_info,
    var,
    levels,
    title,
    "S",
    month,
    TLAT,
    TLON,
    path_HadleyOI,
)

levels = np.array(cice_vars[var][0]["levels"])
title = "September " + cice_vars[var][1]["title"]
month = 9

plot_diff(
    case_names,
    case_info,
    var,
    levels,
    title,
    "S",
    month,
    TLAT,
    TLON,
    path_HadleyOI,
)

In [ ]:
var = "hi"
levels = np.array(cice_vars[var][0]["levels"])
title = "February " + cice_vars[var][1]["title"]
month = 2

plot_diff(
    case_names,
    case_info,
    var,
    levels,
    title,
    "S",
    month,
    TLAT,
    TLON,
    path_HadleyOI,
)

levels = np.array(cice_vars[var][0]["levels"])
title = "September " + cice_vars[var][1]["title"]
month = 9

plot_diff(
    case_names,
    case_info,
    var,
    levels,
    title,
    "S",
    month,
    TLAT,
    TLON,
    path_HadleyOI,
)

In [ ]:
var = "hs"
levels = np.array(cice_vars[var][0]["levels"])
title = "February " + cice_vars[var][1]["title"]
month = 2

plot_diff(
    case_names,
    case_info,
    var,
    levels,
    title,
    "S",
    month,
    TLAT,
    TLON,
    path_HadleyOI,
)

levels = np.array(cice_vars[var][0]["levels"])
title = "September " + cice_vars[var][1]["title"]
month = 9

plot_diff(
    case_names,
    case_info,
    var,
    levels,
    title,
    "S",
    month,
    TLAT,
    TLON,
    path_HadleyOI,
)

In [ ]:
# set up dictionary for all cases listed for just uvel and vvel

cice_hist_vars = ["aice", "uvel", "vvel"]

case_info = {}
for i, (case_name, case_nickname, start_date, end_date, tsdir) in enumerate(
    zip(case_names, case_nicknames, start_dates, end_dates, ts_dir)
):
    # Only open the variables used here; opening every tseries file can exceed
    # xarray's file cache and cause "NetCDF: Not a valid ID" errors.
    ts_files = sorted(
        f
        for var in cice_hist_vars
        for f in glob.glob(
            os.path.join(
                tsdir,
                case_name,
                "ice",
                "proc",
                "tseries",
                f"{case_name}.cice.h.{var}.*.nc",
            )
        )
    )
    ds = xr.open_mfdataset(
        ts_files,
        data_vars="minimal",
        compat="override",
        coords="minimal",
    ).sel(time=slice(start_date, end_date))

    ds_ann = ds.resample(time="YS").mean(dim="time")
    climo_n_years = min(
        climo_nyears, len(ds_ann.time)
    )  ## had to name this something different than global climo_nyears set in params

    case_info[case_name] = {
        "case_nickname": case_nickname,
        "start_date": start_date,
        "end_date": end_date,
        "ds": ds,
        "ds_ann": ds_ann,
        "climo_nyears": climo_n_years,
    }

In [ ]:
vect_diff(
    case_names,
    case_info,
    angle.values,
    "N",
    TLAT.values,
    TLON.values,
)

In [ ]:
vect_diff(
    case_names,
    case_info,
    angle.values,
    "S",
    TLAT.values,
    TLON.values,
)

In [ ]:
if client and not cupid_run:
    client.shutdown()